# Probabilistic Formula 1 Race Forecasting
## From point estimates to coherent finishing-position distributions

This notebook documents the move from predicting one finishing position per driver to forecasting a complete probability distribution over the field. It summarizes the leakage-safe data design, rolling-origin model selection, race-level coherence constraint, frozen 2024 holdout, and final article figures.

### Reproduction modes

The default report mode loads committed metrics and locally generated predictions. Set `RUN_FULL_PIPELINE = True` only when you intentionally want to rebuild the feature store and refit every rolling-origin ordinal model. The full run can take several minutes.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
DATASET = ROOT / 'data_processed/f1_probabilistic_prerace.parquet'
PREDICTIONS = ROOT / 'data_processed/f1_2024_frozen_model_probabilities.parquet'
FIGURE_DIR = ROOT / 'outputs/article_probabilistic_forecasting/final'
RUN_FULL_PIPELINE = False

print('Python:', sys.version.split()[0])
print('Project root:', ROOT)
print('Full pipeline enabled:', RUN_FULL_PIPELINE)

In [ ]:
if RUN_FULL_PIPELINE:
    commands = [
        [sys.executable, 'src/build_probabilistic_dataset.py'],
        [sys.executable, '08_validate_race_balancing.py'],
        [sys.executable, '09_diagnose_distribution_shapes.py'],
        [sys.executable, '09_compare_ordinal_models.py'],
        [sys.executable, '10_freeze_selected_structure.py'],
        [sys.executable, '10_finalize_article_suite.py'],
        [sys.executable, '11_polish_article_visualizations.py'],
    ]
    for command in commands:
        print('Running:', ' '.join(command))
        subprocess.run(command, cwd=ROOT, check=True)
else:
    print('Report mode: using existing frozen outputs.')

## 1. Dataset and temporal design

Each row represents one driver entering one race. All rolling features are shifted, championship standings represent the pre-race state, and prior-season features are stored separately. Development uses expanding-window validation from 2018 through 2023; 2024 is held out until the architecture is frozen.

In [ ]:
df = pd.read_parquet(DATASET)
coverage = pd.Series({
    'rows': len(df),
    'races': df['raceId'].nunique(),
    'drivers': df['driverId'].nunique(),
    'first_season': int(df['year'].min()),
    'last_season': int(df['year'].max()),
    'minimum_position': int(df['finish_position'].min()),
    'maximum_position': int(df['finish_position'].max()),
})
coverage.to_frame('value')

In [ ]:
annual = (
    df.groupby('year')
      .agg(rows=('driverId', 'size'), races=('raceId', 'nunique'),
           drivers=('driverId', 'nunique'), dnf_rate=('is_dnf', 'mean'))
)
annual.style.format({'dnf_rate': '{:.1%}'})

## 2. Why independent thresholds were replaced

The original full model independently estimated each cumulative threshold, `P(finish ≤ k)`. This was predictively competitive but structurally unstable: 87.7% of 2023 rows contained a threshold crossing before monotonic repair, and every resulting driver distribution was multimodal. Sinkhorn balancing corrected competition between drivers, but it could not correct fragmented driver-level shapes.

The proportional-odds challenger uses shared coefficients and ordered cutpoints, preventing cumulative-threshold crossing by construction. Both models are race-balanced before comparison.

In [ ]:
ordinal_summary = pd.read_json(
    ROOT / 'outputs/figures_09/ordinal_model_comparison_summary.json'
).set_index('model')

columns = [
    'rps', 'log_loss', 'expected_position_mae',
    'win_brier', 'podium_brier', 'points_brier',
    'share_multimodal', 'mean_local_peaks'
]
ordinal_summary[columns].style.format('{:.4f}')

In [ ]:
display(Image(filename=str(FIGURE_DIR / '01_model_selection_tradeoff_final.png')))

## 3. Freeze before the holdout

The model specification is written before the challenger is evaluated on 2024. The gate allows a small RPS trade-off only when exact-position calibration, smoothness, and coherence improve materially.

In [ ]:
with open(ROOT / 'outputs/figures_09/frozen_model_specification.json') as file:
    frozen_specification = json.load(file)

pd.json_normalize(frozen_specification).T.rename(columns={0: 'frozen value'})

## 4. Honest 2024 evaluation

The frozen ordinal model achieves the best Ranked Probability Score, expected-position MAE, and points Brier score. Qualifying remains marginally better on exact-position log loss and remains particularly strong for winner and podium events.

In [ ]:
holdout = pd.read_json(
    ROOT / 'outputs/article_probabilistic_forecasting/2024_frozen_model_comparison.json'
).set_index('model')
holdout[[
    'rps', 'log_loss', 'expected_position_mae', 'modal_position_accuracy',
    'win_brier', 'podium_brier', 'points_brier', 'share_multimodal'
]].style.format('{:.4f}')

In [ ]:
display(Image(filename=str(FIGURE_DIR / '02_2024_holdout_comparison_final.png')))

## 5. Coherent race-level probabilities

A valid driver distribution sums to one across positions. A coherent race forecast adds a second constraint: probabilities must also sum to one down each active finishing-position column. The next check verifies both properties for the final 2024 race.

In [ ]:
predictions = pd.read_parquet(PREDICTIONS)
latest_race_id = predictions['raceId'].max()
latest = predictions[predictions['raceId'] == latest_race_id]
matrix = latest.pivot(
    index='driverId', columns='position', values='position_probability'
).fillna(0)
field_size = int(latest['field_size'].iloc[0])
active = matrix.loc[:, 1:field_size]

coherence = pd.Series({
    'maximum driver-row error': float(np.abs(active.sum(axis=1) - 1).max()),
    'maximum position-column error': float(np.abs(active.sum(axis=0) - 1).max()),
    'total win probability': float(active.iloc[:, :1].to_numpy().sum()),
    'total podium probability': float(active.iloc[:, :3].to_numpy().sum()),
    'total points probability': float(active.iloc[:, :10].to_numpy().sum()),
})
coherence.to_frame('value')

In [ ]:
display(Image(filename=str(FIGURE_DIR / '03_smooth_coherent_heatmap_final.png')))

## 6. Reading the forecast

The article suite presents the same frozen matrix in three complementary ways: individual driver distributions, familiar event probabilities, and expected positions with central 80% ranges.

In [ ]:
for filename in [
    '04_driver_distribution_examples_final.png',
    '05_win_podium_points_probabilities_final.png',
    '06_expected_finish_uncertainty_final.png',
]:
    display(Image(filename=str(FIGURE_DIR / filename)))

## Conclusions

1. Point estimates conceal meaningful differences in uncertainty.
2. Independently fitted cumulative thresholds can be accurate while producing internally fragmented distributions.
3. Race balancing imposes the competitive structure of Formula 1 at almost no predictive cost.
4. Proportional odds eliminates threshold crossing and improves the frozen 2024 full-field forecast.
5. Qualifying remains exceptionally informative for exact positions, winners, and podiums.

The next modeling stage should separate DNF/disruption risk from running order, introduce era-aware or recency-weighted training, and compare the ordinal model with a race-ranking model before building a live 2026 forecasting system.